# 15 — PCA por dimensión

Objetivo general del cuaderno:
Vamos a resolver qué hacer con los casos de baja cobertura y valores extremos. Vamos a resolver la estrategia de imputación sobre el resto, estandarizar por dimensión y correr PCA por dimensión con sus diagnósticos correspondientes.

## 1.  importacion de librerias y carga de datos

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option("display.max_columns", None)

In [6]:
IN_PATH = Path("data/processed/14_pais_features.xlsx")  # ajustá la ruta si tu archivo está en otro lugar
df_pais_features = pd.read_excel(IN_PATH, sheet_name="pais_features", index_col=0)

print(f"Shape df_pais_features: {df_pais_features.shape}")
df_pais_features.head()

Shape df_pais_features: (193, 2058)


AG.LND.TOTL.K2_media  BG.GSR.NFSV.GD.ZS_media  \
ISO-alpha3                                                  
AFG                     652230.0                14.416094   
AGO                    1246700.0                19.370284   
ALB                      27400.0                33.667719   
AND                        470.0                81.447556   
ARE                      71020.0                      NaN   

            BM.GSR.CMCP.ZS_media  BM.GSR.FCTY.CD_media  BM.GSR.GNFS.CD_media  \
ISO-alpha3                                                                     
AFG                    48.557755          1.102253e+08          7.302868e+09   
AGO                    35.735548          6.633857e+09          2.690268e+10   
ALB                    13.514135          2.690858e+08          4.870589e+09   
AND                    41.654165          2.187798e+08          1.879695e+09   
ARE                          NaN                   NaN                   NaN   

            BM.GSR.INSF.ZS_media  BM.GSR.MRCH.CD_media  BM.GSR.NFSV.CD_media  \
ISO-alpha3                                                                     
AFG                     2.040682          6.048209e+09          1.254658e+09   
AGO                     4.559638          1.423333e+10          1.266935e+10   
ALB                     4.229901          3.228758e+09          1.641832e+09   
AND                     4.325387          1.348060e+09          5.316355e+08   
ARE                          NaN                   NaN                   NaN   

            BM.GSR.ROYL.CD_media  BM.GSR.TOTL.CD_media  BM.GSR.TRAN.ZS_media  \
ISO-alpha3                                                                     
AFG                 8.508323e+06          7.413093e+09             70.091566   
AGO                 1.008403e+08          3.353654e+10             21.448274   
ALB                 1.758352e+07          5.139675e+09             15.520530   
AND                 3.736637e+05          2.098475e+09             13.758242   
ARE                          NaN                   NaN                   NaN   

            BM.GSR.TRVL.ZS_media  BM.KLT.DINV.CD.WD_media  \
ISO-alpha3                                                  
AFG                     9.129564             7.591159e+06   
AGO                     2.550764             5.048538e+08   
ALB                    66.826756             7.618535e+07   
AND                    29.053041             1.174413e+08   
ARE                          NaN             8.620458e+09   

            BM.KLT.DINV.WD.GD.ZS_media  BM.TRF.PRVT.CD_media  \
ISO-alpha3                                                     
AFG                           0.041417          3.978885e+08   
AGO                           0.635668          5.592257e+08   
ALB                           0.604012          1.522573e+08   
AND                           3.914739          5.778313e+07   
ARE                           2.591342                   NaN   

            BM.TRF.PWKR.CD.DT_media  BN.CAB.XOKA.CD_media  \
ISO-alpha3                                                  
AFG                    3.534705e+08         -2.888089e+09   
AGO                    8.341721e+08          3.021819e+09   
ALB                    1.262605e+08         -1.005074e+09   
AND                    8.764456e+07          5.085633e+08   
ARE                             NaN                   NaN   

            BN.CAB.XOKA.GD.ZS_media  BN.FIN.TOTL.CD_media  \
ISO-alpha3                                                  
AFG                      -15.123911          5.884412e+08   
AGO                        3.098622          1.750814e+09   
ALB                       -8.867333         -7.373159e+08   
AND                       16.768451          5.374083e+08   
ARE                             NaN                   NaN   

            BN.GSR.FCTY.CD_media  BN.GSR.GNFS.CD_media  BN.GSR.MRCH.CD_media  \
ISO-alpha3                                                                     
AFG     

## 2. Vamos a ver cómo se distribuyen los valores faltantes

In [7]:
missingness_pais_completo = (
    df_pais_features.isna().mean(axis=1) * 100
).round(1).sort_values(ascending=True)

df_missingness_pais_ordenado = missingness_pais_completo.reset_index()
df_missingness_pais_ordenado.columns = ["pais", "pct_missingness"]
df_missingness_pais_ordenado["orden"] = range(1, len(df_missingness_pais_ordenado) + 1)

df_regiones = pd.read_excel("df_regiones_miembros_onu.xlsx")  # ajustar ruta si corresponde

df_missingness_pais_ordenado = df_missingness_pais_ordenado.merge(
    df_regiones[["ISO-alpha3", "Member State"]],
    left_on="pais", right_on="ISO-alpha3", how="left"
).drop(columns="ISO-alpha3").rename(columns={"Member State": "nombre_pais"})

df_missingness_pais_ordenado = df_missingness_pais_ordenado[["pais", "nombre_pais", "pct_missingness", "orden"]]

print(f"N paises: {len(df_missingness_pais_ordenado)}")
df_missingness_pais_ordenado.tail(50)

N paises: 193


,pais,nombre_pais,pct_missingness,orden
143,YEM,Yemen,10.2,144
144,GNB,Guinea-Bissau,10.3,145
145,DJI,Djibouti,11.5,146
146,MWI,Malawi,11.5,147
147,STP,Sao Tome and Principe,11.6,148
148,GUY,Guyana,11.9,149
149,BLZ,Belize,12.0,150
150,SYC,Seychelles,12.4,151
151,SYR,Syrian Arab Republic,12.6,152
152,SUR,Suriname,13.0,153


In [8]:
fig_bar = px.bar(
    df_missingness_pais_ordenado, x="orden", y="pct_missingness",
    hover_data=["pais"],
    title="Missingness por país, ordenado de menor a mayor (193 países)",
    labels={"orden": "Países ordenados (de menor a mayor missingness)", "pct_missingness": "% de features faltantes"}
)
fig_bar.show()

In [9]:
fig_line = px.line(
    df_missingness_pais_ordenado, x="orden", y="pct_missingness",
    markers=False,
    title="Curva de missingness por país, ordenada (193 países)",
    labels={"orden": "Países ordenados (de menor a mayor missingness)", "pct_missingness": "% de features faltantes"}
)
fig_line.show()

In [10]:
umbrales_pais = list(range(0, 81, 3))

n_excluidos_por_umbral = [
    (missingness_pais_completo > u).sum()
    for u in umbrales_pais
]

df_sensibilidad_pais = pd.DataFrame({
    "umbral": umbrales_pais,
    "n_excluidos": n_excluidos_por_umbral,
})

fig_sens_pais = px.line(
    df_sensibilidad_pais, x="umbral", y="n_excluidos", markers=True,
    title="Sensibilidad: países excluidos según umbral de missingness",
    labels={"umbral": "Umbral de missingness por país (%)", "n_excluidos": "N países excluidos"}
)
fig_sens_pais.show()

In [11]:
# Tabla de saltos (gaps) entre países consecutivos en la cola alta, para ubicar el valle real
cola_alta = df_missingness_pais_ordenado.tail(30).copy()
cola_alta["salto"] = cola_alta["pct_missingness"].diff()
cola_alta.sort_values("salto", ascending=False).head(10)

,pais,nombre_pais,pct_missingness,orden,salto
191,LIE,Liechtenstein,67.4,192,7.1
189,PRK,Democratic People's Republic of Korea,55.2,190,5.1
190,AND,Andorra,60.3,191,5.1
192,MCO,Monaco,70.9,193,3.5
177,SOM,Somalia,31.8,178,3.2
176,KIR,Kiribati,28.6,177,3.0
188,TUV,Tuvalu,50.1,189,2.9
170,CUB,Cuba,22.2,171,2.9
187,NRU,Nauru,47.2,188,2.8
186,SMR,San Marino,44.4,187,2.7


Boxplot de missingness por país (con detección de outliers)

In [12]:
Q1 = missingness_pais_completo.quantile(0.25)
Q3 = missingness_pais_completo.quantile(0.75)
IQR = Q3 - Q1
umbral_tukey = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.1f}%  |  Q3: {Q3:.1f}%  |  IQR: {IQR:.1f}")
print(f"Umbral de outlier (Tukey, Q3 + 1.5*IQR): {umbral_tukey:.1f}%")

paises_outlier_tukey = missingness_pais_completo[missingness_pais_completo > umbral_tukey]
print(f"\nPaises outlier segun Tukey: {len(paises_outlier_tukey)}")
print(paises_outlier_tukey)

Q1: 1.7%  |  Q3: 10.3%  |  IQR: 8.6
Umbral de outlier (Tukey, Q3 + 1.5*IQR): 23.2%

Paises outlier segun Tukey: 22
ISO-alpha3
ERI    23.4
LBR    23.8
LCA    24.3
SSD    24.9
VCT    25.6
KIR    28.6
SOM    31.8
MHL    33.0
PLW    33.8
TKM    33.9
KNA    36.4
ATG    37.2
DMA    39.1
GRD    41.1
FSM    41.7
SMR    44.4
NRU    47.2
TUV    50.1
PRK    55.2
AND    60.3
LIE    67.4
MCO    70.9
dtype: float64


In [13]:
fig_box_pais = px.box(
    y=missingness_pais_completo, points="all",
    title="Distribución de missingness por país (193 países) — detección de outliers",
    labels={"y": "% de features faltantes"}
)
fig_box_pais.show()

## 3. Exclusión de países con alta missingness (>40%)
Se excluyen los 9 países identificados (microestados extremos + Corea del Norte), según la decisión ya justificada.

In [14]:
paises_excluidos_missingness = ["MCO", "LIE", "AND", "PRK", "TUV", "NRU", "SMR", "FSM", "GRD"]

df_pais_features_reducido = df_pais_features.drop(index=paises_excluidos_missingness, errors="ignore")

print(f"Paises excluidos: {len(paises_excluidos_missingness)}")
print(f"Shape original: {df_pais_features.shape}")
print(f"Shape tras exclusion: {df_pais_features_reducido.shape}")


Paises excluidos: 9
Shape original: (193, 2058)
Shape tras exclusion: (184, 2058)


## 4. Missingness remanente (184 países)
Panorama real sobre el que se decide la imputación.

In [15]:
missingness_col_post = (df_pais_features_reducido.isna().mean() * 100).round(1)
missingness_pais_post = (df_pais_features_reducido.isna().mean(axis=1) * 100).round(1)

print(f"Missingness global promedio (184 paises): {missingness_col_post.mean():.2f}%")
print("\nTop 10 columnas con mayor missingness:")
print(missingness_col_post.sort_values(ascending=False).head(10))


Missingness global promedio (184 paises): 6.81%

Top 10 columnas con mayor missingness:
GB.XPD.RSDV.GD.ZS_tendencia      40.2
GB.XPD.RSDV.GD.ZS_volatilidad    40.2
IP.PAT.RESD_tendencia            34.2
IP.PAT.RESD_volatilidad          34.2
SH.H2O.SMDW.ZS_media             29.9
SH.H2O.SMDW.ZS_volatilidad       29.9
SH.H2O.SMDW.ZS_tendencia         29.9
BX.GSR.ROYL.CD_volatilidad       29.3
BX.GSR.ROYL.CD_tendencia         29.3
GC.TAX.IMPT.ZS_tendencia         28.8
dtype: float64


## 5. Imputación
Se estandarizan las variables y luego se imputa con KNN sobre la matriz ya estandarizada — así la distancia entre países no queda dominada por variables en escalas más grandes (USD, etc.).Se usa un  método KNN con  k=5 . 

In [16]:
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error

col_means = df_pais_features_reducido.mean(skipna=True)
col_stds = df_pais_features_reducido.std(skipna=True)
X_std = (df_pais_features_reducido - col_means) / col_stds

K_CANDIDATOS = [3, 5, 8, 10, 15, 20]
N_REPETICIONES = 5
FRACCION_ENMASCARADA = 0.10
SEMILLA_BASE = 42

celdas_observadas = np.argwhere(X_std.notna().values)

resultados_busqueda = []
for k in K_CANDIDATOS:
    errores_k = []
    for rep in range(N_REPETICIONES):
        rng = np.random.default_rng(SEMILLA_BASE + rep)
        n_mascara = int(len(celdas_observadas) * FRACCION_ENMASCARADA)
        idx_mascara = rng.choice(len(celdas_observadas), size=n_mascara, replace=False)
        filas_mask, cols_mask = celdas_observadas[idx_mascara].T

        X_con_mascara = X_std.values.copy()
        valores_reales = X_con_mascara[filas_mask, cols_mask].copy()
        X_con_mascara[filas_mask, cols_mask] = np.nan

        imputer = KNNImputer(n_neighbors=k)
        X_imputado_prueba = imputer.fit_transform(X_con_mascara)
        valores_imputados = X_imputado_prueba[filas_mask, cols_mask]

        rmse = np.sqrt(mean_squared_error(valores_reales, valores_imputados))
        errores_k.append(rmse)

    resultados_busqueda.append({
        "k": k, "rmse_promedio": np.mean(errores_k), "rmse_sd": np.std(errores_k)
    })
    print(f"k={k}: RMSE promedio={np.mean(errores_k):.4f} (sd={np.std(errores_k):.4f}) sobre {N_REPETICIONES} repeticiones")

df_busqueda_k = pd.DataFrame(resultados_busqueda).sort_values("rmse_promedio")
K_ELEGIDO = int(df_busqueda_k.iloc[0]["k"])
print(f"\nK elegido (menor RMSE): {K_ELEGIDO}")

df_busqueda_k

k=3: RMSE promedio=0.8713 (sd=0.0084) sobre 5 repeticiones
k=5: RMSE promedio=0.8625 (sd=0.0070) sobre 5 repeticiones
k=8: RMSE promedio=0.8660 (sd=0.0065) sobre 5 repeticiones
k=10: RMSE promedio=0.8694 (sd=0.0061) sobre 5 repeticiones
k=15: RMSE promedio=0.8788 (sd=0.0062) sobre 5 repeticiones
k=20: RMSE promedio=0.8862 (sd=0.0065) sobre 5 repeticiones

K elegido (menor RMSE): 5


,k,rmse_promedio,rmse_sd
1,5,0.862475,0.007042
2,8,0.866048,0.006515
3,10,0.869408,0.006116
0,3,0.871324,0.008366
4,15,0.878813,0.006234
5,20,0.886231,0.006469


In [17]:
imputer_final = KNNImputer(n_neighbors=K_ELEGIDO)
X_imputado_array = imputer_final.fit_transform(X_std)
df_imputado = pd.DataFrame(X_imputado_array, index=X_std.index, columns=X_std.columns)

print(f"Shape df_imputado: {df_imputado.shape}")
print(f"NaN remanentes: {df_imputado.isna().sum().sum()}")

Shape df_imputado: (184, 2058)
NaN remanentes: 0


## 6. Mapeo de variables a dimensión analítica
Se reutiliza la clasificación por dimensión ya calculada en el cuaderno 14 si está en memoria; si no, se reconstruye desde `LLM_dataset_final.xlsx` (columna `groq_dimension_principal_original`) como respaldo — **verificar que coincide con el criterio del cuaderno 14 antes de dar este resultado por definitivo.**

In [18]:
if "df_disponibilidad_variables" in dir():
    mapa_dimension = df_disponibilidad_variables.set_index("codigo")["dimension_principal"].to_dict()
    print("Mapeo de dimension_principal tomado de df_disponibilidad_variables (en memoria).")
else:
    df_disponibilidad_variables = pd.read_excel(IN_PATH, sheet_name="disponibilidad_variables")
    mapa_dimension = df_disponibilidad_variables.set_index("codigo")["dimension_principal"].to_dict()
    print("Mapeo de dimension_principal cargado desde la hoja 'disponibilidad_variables' del Excel del cuaderno 14.")

dimension_por_columna = {
    col: mapa_dimension.get(col.rsplit("_", 1)[0], "SIN_CLASIFICAR")
    for col in df_imputado.columns
}
pd.Series(dimension_por_columna).value_counts()

Mapeo de dimension_principal cargado desde la hoja 'disponibilidad_variables' del Excel del cuaderno 14.


SOCIODEMOGRAFICA    774
ESTRUCTURAL         621
MONETARIA           423
INSTITUCIONAL       240
Name: count, dtype: int64

## 7. PCA por dimensión × estadístico (media, tendencia, volatilidad)
Se corre un PCA independiente para cada combinación de dimensión (MONETARIA, INSTITUCIONAL, ESTRUCTURAL, SOCIODEMOGRAFICA) y estadístico de trayectoria (media, tendencia, volatilidad) — 12 PCA en total, en vez de uno solo por dimensión que mezclaba los tres estadísticos. Esto evita la colinealidad entre media/tendencia/volatilidad de una misma variable. Se protegen de la poda automática las variables consideradas teóricamente centrales por dimensión (tasas de interés y dinero amplio en MONETARIA; calidad institucional y Estado de derecho en INSTITUCIONAL; apertura comercial e inversión en ESTRUCTURAL; población y desigualdad en SOCIODEMOGRAFICA), incluso a costa de un KMO algo menor. Se retienen componentes hasta explicar ≥70% de la varianza.

In [19]:
def es_nivel_absoluto(codigo_base):
    """Indicadores cuyo sufijo de unidad WDI es un nivel monetario absoluto
    (USD o moneda local, corriente o constante) — escalan con el tamaño de
    la economia, no con la politica monetaria-fiscal en si."""
    tokens = codigo_base.split(".")
    return any(t in {"CD", "KD", "CN", "KN"} for t in tokens)

In [20]:
from sklearn.decomposition import PCA
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

def ejecutar_pca_bloque(df_datos, cols_bloque, nombre_bloque, varianza_objetivo=0.70):
    X = df_datos[cols_bloque]
    kmo_por_item, kmo_global = calculate_kmo(X)
    chi2, p_valor = calculate_bartlett_sphericity(X)
    pca = PCA()
    scores = pca.fit_transform(X)
    var_exp = pca.explained_variance_ratio_
    var_acum = np.cumsum(var_exp)
    n_comp = max(int(np.searchsorted(var_acum, varianza_objetivo) + 1), 1)
    df_scores = pd.DataFrame(scores[:, :n_comp], index=X.index,
                              columns=[f"{nombre_bloque}_PC{i+1}" for i in range(n_comp)])
    df_cargas = pd.DataFrame(pca.components_[:n_comp].T, index=cols_bloque,
                              columns=[f"{nombre_bloque}_PC{i+1}" for i in range(n_comp)])
    print(f"--- {nombre_bloque} ---")
    print(f"N variables: {len(cols_bloque)} | KMO global: {kmo_global:.3f} "
          f"({'adecuado' if kmo_global >= 0.6 else 'BAJO, revisar'}) | Bartlett p-valor: {p_valor:.4g}")
    print(f"Componentes retenidos (>=70% var.): {n_comp} | Var. acumulada: {var_acum[n_comp-1]*100:.1f}%\n")
    return df_scores, df_cargas, var_exp, kmo_global, p_valor


def eliminar_redundantes(df_datos, columnas, umbral=0.95):
    if len(columnas) < 2:
        return columnas
    corr = df_datos[columnas].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    a_eliminar = set()
    for col in upper.columns:
        if col in a_eliminar:
            continue
        correlacionadas = upper.index[upper[col] > umbral].tolist()
        a_eliminar.update(correlacionadas)
    return [c for c in columnas if c not in a_eliminar]


def podar_hasta_p_menor_n(df_datos, columnas, n_max, umbral_inicial=0.95, paso=0.05, piso=0.20):
    umbral = umbral_inicial
    cols_actuales = columnas
    while len(cols_actuales) >= n_max and umbral > piso:
        cols_actuales = eliminar_redundantes(df_datos, columnas, umbral)
        if len(cols_actuales) >= n_max:
            umbral = round(umbral - paso, 2)
    return cols_actuales, umbral


def seleccionar_por_msa_protegido(df_datos, cols_podables, cols_protegidas,
                                   kmo_objetivo=0.60, min_variables=10, max_iter=500):
    """Poda por MSA calculando siempre el KMO sobre protegidas + podables juntas,
    sacando solo variables NO protegidas. Las protegidas nunca se eliminan."""
    cols_podables = list(cols_podables)
    kmo_global = np.nan
    for _ in range(max_iter):
        cols_actuales = cols_protegidas + cols_podables
        if len(cols_actuales) < 2:
            break
        X = df_datos[cols_actuales]
        kmo_por_item, kmo_global = calculate_kmo(X)
        if not np.isnan(kmo_global) and kmo_global >= kmo_objetivo:
            break
        if len(cols_podables) <= min_variables:
            break
        kmo_series = pd.Series(kmo_por_item, index=cols_actuales)
        kmo_podables = kmo_series[cols_podables]
        if kmo_podables.isna().all():
            cols_podables.pop(0)
        else:
            peor = kmo_podables.idxmin()
            cols_podables.remove(peor)
    return cols_protegidas + cols_podables, kmo_global


CODIGOS_PRECIOS_EXCLUIR_ADICIONAL = ["PA.NUS.PRVT.PLI"]

VARIABLES_PROTEGIDAS = {
    "MONETARIA": ["FR.INR.DPST", "FR.INR.LEND", "FR.INR.RINR",
                  "FM.LBL.BMNY.GD.ZS", "FM.LBL.BMNY.ZG", "FM.LBL.BMNY.CN", "FM.LBL.BMNY.IR.ZS",
                  "FM.AST.CGOV.ZG.M3", "FM.AST.PRVT.ZG.M3", "PA.NUS.FCRF"],
    "INSTITUCIONAL": ["v2x_polyarchy", "GOV_WGI_RQ_SC", "GOV_WGI_RL_SC", "GOV_WGI_CC_SC"],
    "ESTRUCTURAL": ["NE.TRD.GNFS.ZS", "NE.EXP.GNFS.ZS", "NE.IMP.GNFS.ZS", "BX.KLT.DINV.WD.GD.ZS"],
    "SOCIODEMOGRAFICA": ["SP.POP.GROW", "SP.DYN.LE00.IN", "SL.UEM.TOTL.ZS"],
}

ESTADISTICOS = ["media", "tendencia", "volatilidad"]
DIMENSIONES = ["MONETARIA", "INSTITUCIONAL", "ESTRUCTURAL", "SOCIODEMOGRAFICA"]
n_paises = df_imputado.shape[0]

resultados_pca = {}
resumen_poda = []

for dim in DIMENSIONES:
    protegidos_dim = VARIABLES_PROTEGIDAS.get(dim, [])
    for stat in ESTADISTICOS:
        nombre_bloque = f"{dim}_{stat}"
        cols_bloque = [
            c for c, d in dimension_por_columna.items()
            if d == dim and c.endswith(f"_{stat}") and c.rsplit("_", 2)[0] not in CODIGOS_PRECIOS_EXCLUIR_ADICIONAL
        ]
        cols_protegidas = [c for c in cols_bloque if c.rsplit(f"_{stat}", 1)[0] in protegidos_dim]
        cols_podables = [c for c in cols_bloque if c.rsplit(f"_{stat}", 1)[0] not in protegidos_dim]

        if dim == "MONETARIA":
            cols_podables_previo = cols_podables
            cols_podables = [c for c in cols_podables if not es_nivel_absoluto(c.rsplit(f"_{stat}", 1)[0])]
            excluidos_nivel = sorted(set(cols_podables_previo) - set(cols_podables))
            if excluidos_nivel:
                print(f"  [{nombre_bloque}] excluidas por nivel absoluto: {len(excluidos_nivel)} -> {excluidos_nivel}")

        if len(cols_podables) >= n_paises:
            n_objetivo = max(n_paises - len(cols_protegidas) - 10, 30)
            cols_podables, umbral_usado = podar_hasta_p_menor_n(df_imputado, cols_podables, n_objetivo)
        else:
            umbral_usado = "sin poda por correlacion (p<n)"

        cols_final, kmo_final = seleccionar_por_msa_protegido(
            df_imputado, cols_podables, cols_protegidas, kmo_objetivo=0.60, min_variables=15
        )

        print(f"{nombre_bloque}: {len(cols_bloque)} -> {len(cols_final)} variables "
              f"(corr {umbral_usado}, KMO final={kmo_final:.3f}, protegidas={len(cols_protegidas)})")
        resumen_poda.append({"bloque": nombre_bloque, "n_original": len(cols_bloque),
                              "n_final": len(cols_final), "kmo_final": round(kmo_final, 3)})

        resultados_pca[nombre_bloque] = ejecutar_pca_bloque(df_imputado, cols_final, nombre_bloque)

df_pca_scores = pd.concat([resultados_pca[b][0] for b in resultados_pca], axis=1)
print(f"\nShape matriz de scores PCA (12 bloques): {df_pca_scores.shape}")
pd.DataFrame(resumen_poda)

  [MONETARIA_media] excluidas por nivel absoluto: 77 -> ['BM.GSR.TOTL.CD_media', 'BM.KLT.DINV.CD.WD_media', 'BM.TRF.PWKR.CD.DT_media', 'BN.KAC.EOMS.CD_media', 'BN.KLT.DINV.CD_media', 'BN.KLT.PTXL.CD_media', 'BN.RES.INCL.CD_media', 'BX.KLT.DINV.CD.WD_media', 'BX.PEF.TOTL.CD.WD_media', 'FI.RES.TOTL.CD_media', 'FI.RES.XGLD.CD_media', 'FM.AST.DOMS.CN_media', 'FM.AST.NFRG.CN_media', 'GC.REV.GOTR.CN_media', 'GC.REV.XGRT.CN_media', 'GC.TAX.GSRV.CN_media', 'GC.TAX.IMPT.CN_media', 'GC.TAX.INTT.CN_media', 'GC.TAX.OTHR.CN_media', 'GC.TAX.TOTL.CN_media', 'GC.TAX.YPKG.CN_media', 'GC.XPN.COMP.CN_media', 'GC.XPN.GSRV.CN_media', 'GC.XPN.INTP.CN_media', 'GC.XPN.OTHR.CN_media', 'GC.XPN.TOTL.CN_media', 'GC.XPN.TRFT.CN_media', 'NE.CON.GOVT.CD_media', 'NE.CON.GOVT.CN_media', 'NE.CON.GOVT.KD.ZG_media', 'NE.CON.GOVT.KD_media', 'NE.CON.GOVT.KN_media', 'NE.CON.TOTL.CD_media', 'NE.CON.TOTL.CN_media', 'NE.CON.TOTL.KD.ZG_media', 'NE.CON.TOTL.KD_media', 'NE.CON.TOTL.KN_media', 'NE.DAB.TOTL.CD_media', 'NE.DAB.TOTL.

c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_ana

--- INSTITUCIONAL_tendencia ---
N variables: 80 | KMO global: 0.805 (adecuado) | Bartlett p-valor: nan
Componentes retenidos (>=70% var.): 13 | Var. acumulada: 71.2%

INSTITUCIONAL_volatilidad: 80 -> 80 variables (corr sin poda por correlacion (p<n), KMO final=0.847, protegidas=3)
--- INSTITUCIONAL_volatilidad ---
N variables: 80 | KMO global: 0.847 (adecuado) | Bartlett p-valor: 0
Componentes retenidos (>=70% var.): 14 | Var. acumulada: 71.5%



c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:199: RuntimeWarning: invalid value encountered in sqrt
  Is = np.sqrt(1 / np.diag(m))
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:199: RuntimeWarning: invalid value encountered in sqrt
  Is = np.sqrt(1 / np.diag(m))
c:\Users\Carmela\A

ESTRUCTURAL_media: 207 -> 120 variables (corr 0.95, KMO final=0.614, protegidas=4)
--- ESTRUCTURAL_media ---
N variables: 120 | KMO global: 0.614 (adecuado) | Bartlett p-valor: nan
Componentes retenidos (>=70% var.): 16 | Var. acumulada: 70.8%



...

SOCIODEMOGRAFICA_tendencia: 258 -> 139 variables (corr 0.9, KMO final=0.605, protegidas=3)
--- SOCIODEMOGRAFICA_tendencia ---
N variables: 139 | KMO global: 0.605 (adecuado) | Bartlett p-valor: 0
Componentes retenidos (>=70% var.): 21 | Var. acumulada: 70.2%



c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_ana

SOCIODEMOGRAFICA_volatilidad: 258 -> 142 variables (corr 0.9, KMO final=0.604, protegidas=3)
--- SOCIODEMOGRAFICA_volatilidad ---
N variables: 142 | KMO global: 0.604 (adecuado) | Bartlett p-valor: 0
Componentes retenidos (>=70% var.): 20 | Var. acumulada: 70.7%


Shape matriz de scores PCA (12 bloques): (184, 173)


c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_analyzer\utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
c:\Users\Carmela\AppData\Local\Programs\Python\Python312\Lib\site-packages\factor_ana

,bloque,n_original,n_final,kmo_final
0,MONETARIA_media,140,63,0.701
1,MONETARIA_tendencia,140,60,0.601
2,MONETARIA_volatilidad,140,63,0.719
3,INSTITUCIONAL_media,80,80,0.915
4,INSTITUCIONAL_tendencia,80,80,0.805
5,INSTITUCIONAL_volatilidad,80,80,0.847
6,ESTRUCTURAL_media,207,120,0.614
7,ESTRUCTURAL_tendencia,207,118,0.603
8,ESTRUCTURAL_volatilidad,207,156,0.611
9,SOCIODEMOGRAFICA_media,258,124,0.796


## 8. Interpretación de cargas factoriales (PC1 y PC2 por bloque)
Variables con mayor peso absoluto en los dos primeros componentes de cada uno de los 12 bloques (dimensión × estadístico) — necesario para nombrar/interpretar conceptualmente cada componente. Ningún componente se acepta sin sentido teórico.

In [21]:
for nombre_bloque, (scores, cargas, var_exp, kmo, p) in resultados_pca.items():
    print(f"=== {nombre_bloque} — Top cargas PC1 ===")
    print(cargas.iloc[:, 0].abs().sort_values(ascending=False).head(8))
    if cargas.shape[1] > 1:
        print(f"\n=== {nombre_bloque} — Top cargas PC2 ===")
        print(cargas.iloc[:, 1].abs().sort_values(ascending=False).head(8))
    print("\n" + "="*60 + "\n")


=== MONETARIA_media — Top cargas PC1 ===
SH.XPD.GHED.GD.ZS_media    0.272050
FD.AST.PRVT.GD.ZS_media    0.253640
FM.AST.PRVT.GD.ZS_media    0.253343
FS.AST.PRVT.GD.ZS_media    0.252742
SH.XPD.GHED.CH.ZS_media    0.240002
PA.NUS.GDP.PLI_media       0.236724
SH.XPD.GHED.GE.ZS_media    0.213071
GC.XPN.TOTL.GD.ZS_media    0.203883
Name: MONETARIA_media_PC1, dtype: float64

=== MONETARIA_media — Top cargas PC2 ===
NY.ADJ.ICTR.GN.ZS_media    0.350331
NY.GNS.ICTR.GN.ZS_media    0.344902
NY.GDS.TOTL.ZS_media       0.323475
NE.CON.TOTL.ZS_media       0.323475
NY.ADJ.NNAT.GN.ZS_media    0.316282
NE.DAB.TOTL.ZS_media       0.270411
NY.GNS.ICTR.ZS_media       0.253107
SH.XPD.CHEX.GD.ZS_media    0.201565
Name: MONETARIA_media_PC2, dtype: float64


=== MONETARIA_tendencia — Top cargas PC1 ===
NY.GNS.ICTR.ZS_tendencia       0.338512
GC.REV.XGRT.GD.ZS_tendencia    0.302465
GC.TAX.TOTL.GD.ZS_tendencia    0.294293
GC.NLD.TOTL.GD.ZS_tendencia    0.292821
NY.ADJ.NNAT.GN.ZS_tendencia    0.283426
NY.ADJ.ICT

In [22]:
for dim in ["INSTITUCIONAL", "ESTRUCTURAL", "SOCIODEMOGRAFICA"]:
    nombre_bloque = f"{dim}_media"
    scores, cargas, var_exp, kmo, p = resultados_pca[nombre_bloque]
    print(f"=== {nombre_bloque} — PC1, mayor carga POSITIVA ===")
    print(cargas.iloc[:, 0].sort_values(ascending=False).head(5))
    print(f"\n=== {nombre_bloque} — PC1, mayor carga NEGATIVA ===")
    print(cargas.iloc[:, 0].sort_values(ascending=True).head(5))
    print("\n" + "="*60 + "\n")

=== INSTITUCIONAL_media — PC1, mayor carga POSITIVA ===
GOV_WGI_VA_SC_media     0.157129
GOV_WGI_VA_EST_media    0.157129
v2x_regime_amb_media    0.145854
GOV_WGI_RL_SC_media     0.142822
GOV_WGI_RL_EST_media    0.142822
Name: INSTITUCIONAL_media_PC1, dtype: float64

=== INSTITUCIONAL_media — PC1, mayor carga NEGATIVA ===
v2xnp_pres_media       -0.143512
v2x_pubcorr_media      -0.134935
v2xpe_exlsocgr_media   -0.134480
v2xpe_exlpol_media     -0.133924
v2x_execorr_media      -0.133531
Name: INSTITUCIONAL_media_PC1, dtype: float64


=== ESTRUCTURAL_media — PC1, mayor carga POSITIVA ===
SL.EMP.WORK.ZS_media       0.178416
FB.ATM.TOTL.P5_media       0.168465
SL.GDP.PCAP.EM.KD_media    0.168107
NV.SRV.EMPL.KD_media       0.162885
SL.IND.EMPL.MA.ZS_media    0.162267
Name: ESTRUCTURAL_media_PC1, dtype: float64

=== ESTRUCTURAL_media — PC1, mayor carga NEGATIVA ===
SL.AGR.EMPL.ZS_media         -0.173935
NV.AGR.TOTL.ZS_media         -0.167967
SH.XPD.EHEX.CH.ZS_media      -0.139595
TM.TAX.MRCH.S

## Exportamos

In [23]:
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "15_pca_scores.xlsx"
df_pca_scores.to_excel(out_path, sheet_name="pca_scores")
print(f"Exportado: {out_path}")

Exportado: data\processed\15_pca_scores.xlsx


In [24]:
import pandas as pd
import numpy as np
from pathlib import Path

filas = []
for nombre_bloque, (scores, cargas, var_exp, kmo, p_valor) in resultados_pca.items():
    var_acum = np.cumsum(var_exp)
    n_retenidos = scores.shape[1]
    for i, (ve, va) in enumerate(zip(var_exp, var_acum), start=1):
        filas.append({
            "bloque": nombre_bloque,
            "componente": i,
            "var_explicada": ve,
            "var_acumulada": va,
            "retenido_original_70pct": i <= n_retenidos,
        })

df_varianza_pca = pd.DataFrame(filas)

OUT_PATH_VARIANZA = Path("data/processed/15_varianza_explicada_por_bloque.xlsx")
df_varianza_pca.to_excel(OUT_PATH_VARIANZA, index=False)
print(f"Exportado: {OUT_PATH_VARIANZA}")
print(f"Bloques: {df_varianza_pca['bloque'].nunique()} | Filas totales: {len(df_varianza_pca)}")

Exportado: data\processed\15_varianza_explicada_por_bloque.xlsx
Bloques: 12 | Filas totales: 1225


In [26]:
filas_completas = []
for nombre_bloque, (scores, cargas, var_exp, kmo, p_valor) in resultados_pca.items():
    n_comp = scores.shape[1]
    var_acum_pct = np.cumsum(var_exp)[n_comp - 1] * 100
    filas_completas.append({
        "bloque": nombre_bloque,
        "componentes": n_comp,
        "var_acum_pct": round(var_acum_pct, 1),
    })

df_componentes = pd.DataFrame(filas_completas)

df_resumen_pca_completo = pd.DataFrame(resumen_poda).merge(df_componentes, on="bloque")
df_resumen_pca_completo

,bloque,n_original,n_final,kmo_final,componentes,var_acum_pct
0,MONETARIA_media,140,63,0.701,11,71.7
1,MONETARIA_tendencia,140,60,0.601,13,71.4
2,MONETARIA_volatilidad,140,63,0.719,12,70.8
3,INSTITUCIONAL_media,80,80,0.915,5,70.3
4,INSTITUCIONAL_tendencia,80,80,0.805,13,71.2
5,INSTITUCIONAL_volatilidad,80,80,0.847,14,71.5
6,ESTRUCTURAL_media,207,120,0.614,16,70.8
7,ESTRUCTURAL_tendencia,207,118,0.603,19,70.2
8,ESTRUCTURAL_volatilidad,207,156,0.611,19,70.4
9,SOCIODEMOGRAFICA_media,258,124,0.796,10,71.0
